### Need to run `logit_lens_metric_gemma.py` (or `logit_lens_metric_qwen.py`) first.
### `basic_shapes_TEST` / `squiggles_30_TEST` (etc.) are already generated and committed to the repo via `generate_dataset.py` — no need to regenerate unless you want a new sample.

In [ ]:
import pickle
import os

from data_class import VisionLanguageDataset

# "basic_shapes_TEST"  = known shapes (circle/triangle/...), words already exist
# "squiggles_30_TEST"  = novel/unknown shapes, no established word -- analogous to
#                         face_recognition's known/unknown split
DATASET_NAME = "basic_shapes_TEST"
MODEL_PATH = "google/gemma-3-12b-it"

target_dataset = VisionLanguageDataset(DATASET_NAME)

logit_lens_output_dir = "logit_lens_results/GEMMA"
file_path = f"{logit_lens_output_dir}/dataset{DATASET_NAME}_model{MODEL_PATH.replace('/', '_')}_all_data.pkl"

with open(file_path, "rb") as f:
    target_logit_lens_data = pickle.load(f)

In [ ]:
data_idx = 36
option_idx = 0   # which candidate position to trace: 0=A, 1=B, 2=C, 3=D
SKIP_LAYERS = 0  # tune once you've seen how many layers this model/pkl has


save_dir = f"{DATASET_NAME}_logit_lens_plots_item{data_idx}_option{option_idx}"
os.makedirs(save_dir, exist_ok=True)

# target_logit_lens_data[data_idx] is a list over layers; each layer entry is a list
# of 4 option entries (one per A/B/C/D bbox); each of those is a list of (word, prob)
# tuples, one per image-patch token that falls inside that option's bounding box.
logit_lens_data = [layer_data[option_idx] for layer_data in target_logit_lens_data[data_idx]]

In [ ]:
from PIL import Image

item = target_dataset[data_idx]

# 2D-shape items don't have per-entity crops like the face dataset -- all 4 candidate
# shapes live inside a single tgt_image, at the bbox given by tgt_pixel_positions[option_idx].
item["ref_image"].save(save_dir + "/ref.png")

tgt_image = item["tgt_image"]
tgt_image.save(save_dir + "/tgt.png")

# Crop out just the shape being traced, for a closer look
(x_min, y_min), (x_max, y_max) = item["tgt_pixel_positions"][option_idx]
tgt_image.crop((x_min, y_min, x_max, y_max)).save(save_dir + "/option_crop.png")

print("shape_types:", item["shape_types"], "| correct answer:", item["answer"])

In [30]:

# ── Find which stems have 2+ consecutive occurrences ─────────────────────────
def consecutive_run_lengths(stems):
    """Return a list of run-length for each index (how long its consecutive run is)."""
    n = len(stems)
    run_len = [1] * n
    i = 0
    while i < n:
        j = i + 1
        while j < n and stems[j] == stems[i]:
            j += 1
        length = j - i
        for k in range(i, j):
            run_len[k] = length
        i = j
    return run_len

In [31]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from nltk.stem import PorterStemmer

# Global font size knob
FONTSIZE = 24
TITLE_FONTSIZE = FONTSIZE + 6
LABEL_FONTSIZE = FONTSIZE + 0
TICK_FONTSIZE = FONTSIZE
ANNOT_FONTSIZE = FONTSIZE
LEGEND_FONTSIZE = FONTSIZE

_stemmer = PorterStemmer()
def stem(word):
    return _stemmer.stem(word.lower())


In [ ]:
import os
import warnings

# Face-specific token indices don't apply here -- leave empty, or after a first run
# fill in a few target_token_idx values (patch positions) you want printed for inspection.
to_show = []

warnings.filterwarnings("ignore", message=r"Glyph .* missing from font")
warnings.filterwarnings("ignore", message=r"Matplotlib currently does not support .* natively")

for target_token_idx in range(len(logit_lens_data[0])):
    layers, words, values = [], [], []

    for_print = ""
    for layer in range(len(logit_lens_data)):
        if layer < SKIP_LAYERS:
            continue
        word, value = logit_lens_data[layer][target_token_idx]
        layers.append(layer)
        words.append(word)
        values.append(value)

        for_print += f"{word} ({value:.2f}), "

    stems = [stem(w) for w in words]

    # print("target_token_idx: ", target_token_idx)
    if target_token_idx in to_show:
        print(f"Patch {target_token_idx} words: {for_print}")
        print()

    run_lengths = consecutive_run_lengths(stems)

    annotate_indices = set()
    i = 0
    while i < len(stems):
        j = i + 1
        while j < len(stems) and stems[j] == stems[i]:
            j += 1
        span_len = j - i
        if span_len == 1:
            annotate_indices.add(i)
        else:
            for idx in range(i, j, 4):
                annotate_indices.add(idx)
        i = j

    GREY = "#CCCCCC"

    # ── Color palettes ────────────────────────────────────────────────────────────
    BG_COLORS    = ["#F08080","#87CEEB","#F5DEB3","#90EE90","#DDA0DD","#FFB347"]
    POINT_COLORS = ["#E05555","#5B8DB8","#6DAE81","#9B72AA","#D4A843","#4DBECC"]

    # Only assign colors to stems that appear in runs of 2+
    qualified_stems = list(dict.fromkeys(s for s, r in zip(stems, run_lengths) if r >= 2))
    qualified_words = list(dict.fromkeys(w for w, r in zip(words, run_lengths) if r >= 2))
    stem_color = {s: BG_COLORS[i % len(BG_COLORS)]      for i, s in enumerate(qualified_stems)}
    word_color = {w: POINT_COLORS[i % len(POINT_COLORS)] for i, w in enumerate(qualified_words)}

    def get_stem_color(i): return stem_color.get(stems[i], GREY)
    def get_word_color(i): return word_color.get(words[i], GREY)

    # ── Plot ──────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(16, 7))

    # Background bands — grey if run < 2, colored otherwise
    i = 0
    while i < len(stems):
        j = i + 1
        while j < len(stems) and stems[j] == stems[i]:
            j += 1
        x0 = layers[i]   - (0.5 if i == 0           else (layers[i]   - layers[i-1]) / 2)
        x1 = layers[j-1] + (0.5 if j == len(layers) else (layers[j]   - layers[j-1]) / 2)
        color = get_stem_color(i)
        ax.axvspan(x0, x1, alpha=0.25, color=color, linewidth=0, zorder=0)
        i = j


    # Line + markers + labels
    ax.plot(layers, values, color="#888888", linewidth=1.8, zorder=2)

    v_loc = 60
    for idx, (layer, word, value) in enumerate(zip(layers, words, values)):
        ax.plot(layer, value, "o", markersize=16, color=get_word_color(idx),
                markeredgecolor="white", markeredgewidth=1.8, zorder=3)
        if idx in annotate_indices:
            # print(f"annotating {word} at layer {layer}, {idx}")
            
            ax.annotate(
                f"{word}",  # no probability
                xy=(layer, value),
                xytext=(0, v_loc),
                textcoords="offset points",
                ha="center", va="bottom" if idx % 2 == 0 else "top",
                fontsize=ANNOT_FONTSIZE,
                # fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35", facecolor=get_stem_color(idx),
                        edgecolor="#999999", alpha=0.9, linewidth=0.8),
                arrowprops=dict(arrowstyle="-|>", color="#777777", lw=0.9),
                zorder=4,
            )

            v_loc = -v_loc


    # Legend (only colored/qualified stems)
    ax.legend(
        handles=[mpatches.Patch(facecolor=stem_color[s], alpha=0.6, label=f"{s}")
                for s in qualified_stems],
        fontsize=LEGEND_FONTSIZE,
    )

    # Axes labels, title, limits
    ax.set(
        xlabel="Layer",
        ylabel="Confidence Score",
        title=f"Patch {target_token_idx} — Logit Lens Across Layers",
        xlim=(layers[0] - 0.8, layers[-1] + 0.8),
        ylim=(-0.08, 1.25),
    )

    # Apply font sizes
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.title.set_size(TITLE_FONTSIZE)
    ax.xaxis.label.set_size(LABEL_FONTSIZE)
    ax.yaxis.label.set_size(LABEL_FONTSIZE)

    # Only show every 5th x tick (plus last if needed)
    if len(layers) > 0:
        xticks = [layers[i] for i in range(0, len(layers), 5)]
        if layers[-1] not in xticks:
            xticks.append(layers[-1])
        ax.set_xticks(xticks)

    ax.grid(alpha=0.25, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    plt.ylim(-0.5, 1.50)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/patch{target_token_idx}.pdf", dpi=100, bbox_inches="tight")
    plt.close()
    # plt.show()